In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Attention, TimeDistributed
from tensorflow.keras.models import Model
import numpy as np

# Generate dummy data
max_doc_length = 10
max_sent_length = 20
vocab_size = 10000
num_samples = 1000

train_doc_data = np.random.randint(vocab_size, size=(num_samples, max_doc_length, max_sent_length))
train_word_data = np.random.randint(vocab_size, size=(num_samples, max_sent_length, vocab_size))
train_labels = np.random.randint(2, size=(num_samples, 1))

test_doc_data = np.random.randint(vocab_size, size=(num_samples // 2, max_doc_length, max_sent_length))
test_word_data = np.random.randint(vocab_size, size=(num_samples // 2, max_sent_length, vocab_size))
test_labels = np.random.randint(2, size=(num_samples // 2, 1))

# Define HAN model
doc_input = Input(shape=(max_doc_length, max_sent_length))
word_input = Input(shape=(max_sent_length, vocab_size))

doc_encoder = LSTM(64, return_sequences=True)(doc_input)
word_attention = TimeDistributed(Dense(1))(word_input)
word_attention = tf.keras.layers.Softmax(axis=-1)(word_attention)
word_representations = tf.keras.layers.Multiply()([word_attention, word_input])
sentence_representations = tf.keras.layers.LSTM(64, return_sequences=True)(word_representations)  # Fix input shape

sentence_encoder = LSTM(64)(sentence_representations)  # Remove return_sequences=True
doc_attention = Attention()([doc_encoder, sentence_encoder])
doc_representations = tf.keras.layers.Multiply()([doc_attention, doc_encoder])

outputs = Dense(1, activation='sigmoid')(doc_representations)

model = Model(inputs=[doc_input, word_input], outputs=outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit([train_doc_data, train_word_data], train_labels, epochs=3, batch_size=32, validation_split=0.2)

# Evaluate the model
loss, accuracy = model.evaluate([test_doc_data, test_word_data], test_labels)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)


Epoch 1/3
25/25 [==============================] - 22s 612ms/step - loss: 0.6973 - accuracy: 0.5139 - val_loss: 0.6895 - val_accuracy: 0.5460
Epoch 2/3
25/25 [==============================] - 16s 650ms/step - loss: 0.6937 - accuracy: 0.5208 - val_loss: 0.6984 - val_accuracy: 0.4840
Epoch 3/3
16/16 [==============================] - 3s 160ms/step - loss: 0.6939 - accuracy: 0.5080
Test Loss: 0.6938596963882446
Test Accuracy: 0.5080000162124634
